# 실습 3. 시연에서 배우기 — 행동 복제 · DAgger · 미니 VLA

**피지컬 AI 실습 시리즈 3/3** · CPU로 충분합니다 (PyTorch 사용).

보상을 정의하기 어려운 작업(빨래 개기, 요리, 조립)은 **사람이 직접 시연**하고 로봇이 따라 배우게 하는 것이 요즘 피지컬 AI의 주류입니다.
실습 1에서 만든 2-링크 로봇팔을 다시 사용합니다.

| 단계 | 내용 | 핵심 질문 |
|---|---|---|
| 3-1 | 전문가 시연 수집 | 텔레오퍼레이션 데이터는 어떻게 생겼나? |
| 3-2 | 행동 복제 (Behavior Cloning) | 지도학습만으로 정책을 배울 수 있나? |
| 3-3 | 분포 이동 (Covariate Shift) | 누가 팔을 툭 치면 왜 망가지나? |
| 3-4 | DAgger | 실수한 상황을 데이터로 만들면? |
| 3-5 | 미니 VLA | "빨간 공으로 가" — 언어로 로봇에게 지시하기 |

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

torch.manual_seed(0)
np.set_printoptions(precision=3, suppress=True)

L1, L2 = 1.0, 0.8
MAX_DQ = 0.08               # 한 스텝에 움직일 수 있는 최대 관절 각도 [rad]
T = 80                      # 에피소드 길이 (스텝)
HOME = np.array([0.0, np.pi / 2])
SUCCESS_TOL = 0.06          # 목표와 손끝 거리가 이 이하면 성공 [m]
PUSH_T, PUSH = 50, 1.5      # 외란: 50번째 스텝에 관절을 최대 ±1.5 rad 밀침


def fk(q):
    """손끝 위치"""
    return np.array([L1 * np.cos(q[0]) + L2 * np.cos(q[0] + q[1]),
                     L1 * np.sin(q[0]) + L2 * np.sin(q[0] + q[1])])


def ik(target):
    """팔꿈치-위 해석적 IK (실습 1과 동일)"""
    x, y = target
    c2 = np.clip((x**2 + y**2 - L1**2 - L2**2) / (2 * L1 * L2), -1, 1)
    s2 = -np.sqrt(1 - c2**2)
    return np.array([np.arctan2(y, x) - np.arctan2(L2 * s2, L1 + L2 * c2), np.arctan2(s2, c2)])


def wrap(a):
    return (a + np.pi) % (2 * np.pi) - np.pi


def sample_target(rng):
    """책상 위(위쪽 반평면)의 닿을 수 있는 지점"""
    r, phi = rng.uniform(0.6, 1.6), rng.uniform(0.15 * np.pi, 0.85 * np.pi)
    return np.array([r * np.cos(phi), r * np.sin(phi)])


# ---- 영상 도구 ----
def arm_points(q):
    """어깨, 팔꿈치, 손끝 좌표"""
    elbow = np.array([L1 * np.cos(q[0]), L1 * np.sin(q[0])])
    return np.stack([np.zeros(2), elbow, fk(q)])


def show_arms(q_paths, titles, goals, objects=None, push_t=None, hold=20, fps=12):
    """여러 팔의 움직임을 나란히 재생합니다.
    q_paths: 패널마다 관절 각도 기록 (스텝 수 x 2) · goals: 패널마다 가야 할 목표
    objects: 패널마다 [(위치, 색), ...] (미니 VLA용) · push_t: 이 스텝에 'PUSH!' 표시"""
    fig, axes = plt.subplots(1, len(q_paths), figsize=(4.2 * len(q_paths), 4.6))
    axes = np.atleast_1d(axes)
    parts = []
    for j, (ax, qs, title, goal) in enumerate(zip(axes, q_paths, titles, goals)):
        if objects is not None:
            for p, c in objects[j]:
                ax.add_patch(plt.Circle(p, 0.12, color=c, alpha=0.9))
            ax.add_patch(plt.Circle(goal, 0.2, fill=False, ls="--", lw=2, color="k"))   # 가야 할 물체
        else:
            ax.plot(*goal, "*", color="gold", mec="k", ms=20)
        trail, = ax.plot([], [], color="tab:orange", lw=1.5, alpha=0.8)
        arm, = ax.plot([], [], "-o", lw=6, ms=9, color="#3a3f47")
        msg = ax.text(0, -1.75, "", ha="center", fontsize=14, fontweight="bold")
        ax.set_title(title, fontsize=11)
        ax.set_xlim(-1.9, 1.9); ax.set_ylim(-1.9, 1.9); ax.set_aspect("equal"); ax.axis("off")
        ax.add_patch(plt.Circle((0, 0), L1 + L2, fill=False, ls=":", color="lightgray"))   # 팔이 닿는 범위
        hand_path = np.array([fk(q) for q in qs])
        parts.append((arm, trail, msg, qs, hand_path, goal))
    n = max(len(p[3]) for p in parts)

    def update(i):
        for arm, trail, msg, qs, hp, goal in parts:
            k = min(i, len(qs) - 1)
            pts = arm_points(qs[k])
            arm.set_data(pts[:, 0], pts[:, 1])
            trail.set_data(hp[:k + 1, 0], hp[:k + 1, 1])
            if push_t is not None and push_t <= k < push_t + 6:
                msg.set_text("PUSH!"); msg.set_color("purple")
            elif i >= n - 1 or k == len(qs) - 1:
                ok = np.linalg.norm(hp[-1] - goal) < SUCCESS_TOL
                msg.set_text("SUCCESS" if ok else "MISS"); msg.set_color("green" if ok else "red")
            else:
                msg.set_text("")

    ani = animation.FuncAnimation(fig, update, frames=n + hold, interval=1000 / fps)
    plt.close(fig)
    try:
        html = ani.to_html5_video()   # Colab에는 ffmpeg가 있어 가벼운 mp4로 만듦
    except Exception:
        html = ani.to_jshtml()        # ffmpeg가 없으면 프레임 묶음으로 대신 보여 줌
    display(HTML(html))

## 3-1. 전문가 시연 수집

실제로는 사람이 조이스틱·VR·리더 암(leader arm)으로 로봇을 조종해 데이터를 모읍니다 (텔레오퍼레이션).
여기서는 IK + 비례 제어로 만든 **전문가(expert)** 가 사람 역할을 대신합니다.

- **관측**: 관절 각도, 손끝 위치, 목표 위치 (숫자 8개)
- **행동**: 다음 스텝에 관절 두 개를 각각 몇 도 더 돌릴지 (숫자 2개)

In [ ]:
def expert_action(q, target):
    return np.clip(0.25 * wrap(ik(target) - q), -MAX_DQ, MAX_DQ)


def make_obs(q, target):
    return np.concatenate([np.cos(q), np.sin(q), fk(q), target]).astype(np.float32)


def rollout(policy, target, rng, disturb=False):
    """policy(q, target) -> action 으로 한 에피소드 실행. disturb=True면 중간에 팔을 툭 침."""
    q = HOME + rng.normal(0, 0.05, 2)
    obs_list, q_list = [], [q.copy()]
    for t in range(T):
        if disturb and t == PUSH_T:
            q = q + rng.uniform(-PUSH, PUSH, 2)        # 외부 충격!
        obs_list.append(make_obs(q, target))
        q = q + np.clip(policy(q, target), -MAX_DQ, MAX_DQ)
        q_list.append(q.copy())
    return np.array(obs_list), np.array(q_list), np.linalg.norm(fk(q) - target)


def collect_demos(n, rng, disturb=False):
    X, Y = [], []
    for _ in range(n):
        target = sample_target(rng)
        obs, qs, _ = rollout(expert_action, target, rng, disturb=disturb)
        X.append(obs)
        Y.append(np.array([expert_action(q, target) for q in qs[:-1]]))
    return np.concatenate(X), np.concatenate(Y).astype(np.float32)


rng = np.random.default_rng(0)
X_demo, Y_demo = collect_demos(30, rng)
print("demo dataset:", X_demo.shape, Y_demo.shape, "(30 episodes x 80 steps)")

fig, ax = plt.subplots(figsize=(5, 5))
for i in range(10):
    tgt = sample_target(rng)
    _, qs, _ = rollout(expert_action, tgt, rng)
    hp = np.array([fk(q) for q in qs])
    ax.plot(hp[:, 0], hp[:, 1], lw=1); ax.plot(*tgt, "r*", ms=10)
ax.plot(*fk(HOME), "ko", ms=8, label="home")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(); ax.set_title("Expert demonstrations (hand paths)")
plt.show()

### 🎬 시연 영상
전문가가 노란 별(목표)로 팔을 뻗는 시연 세 개입니다. 로봇이 배울 데이터가 바로 이 움직임입니다.

In [ ]:
rng_demo = np.random.default_rng(3)
demo_targets = [sample_target(rng_demo) for _ in range(3)]
show_arms([rollout(expert_action, t, rng_demo)[1] for t in demo_targets],
          ["demo 1", "demo 2", "demo 3"], demo_targets)

## 3-2. 행동 복제 (Behavior Cloning)

아이디어는 단순합니다: **"이 상황에서 전문가는 이렇게 했다"** 를 지도학습으로 외웁니다.

문제집에 비유하면: 문제(상황)와 정답(전문가의 행동)을 잔뜩 모아 두고, 신경망의 답이 정답과 비슷해지도록 반복해서 풀게 합니다.
이미지 분류를 학습시키는 방법과 똑같습니다.

ALOHA의 ACT, Diffusion Policy, 그리고 VLA 모델들도 큰 틀에서는 이 방식(시연 데이터로 지도학습)에 기반합니다.

In [ ]:
class MLPPolicy(nn.Module):
    def __init__(self, obs_dim=8, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 2), nn.Tanh())

    def forward(self, obs):
        return self.net(obs) * MAX_DQ


def train_bc(X, Y, steps=1500, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    model = MLPPolicy(X.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, Yt = torch.from_numpy(X), torch.from_numpy(Y)
    for _ in range(steps):
        idx = torch.randint(0, len(Xt), (256,))
        loss = ((model(Xt[idx]) - Yt[idx]) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return model, loss.item()


def as_policy(model):
    def policy(q, target):
        with torch.no_grad():
            return model(torch.from_numpy(make_obs(q, target))).numpy()
    return policy


def success_rate(policy, n=100, disturb=False, seed=123):
    rng = np.random.default_rng(seed)
    d = np.array([rollout(policy, sample_target(rng), rng, disturb=disturb)[2] for _ in range(n)])
    return (d < SUCCESS_TOL).mean(), np.median(d)


bc_model, final_loss = train_bc(X_demo, Y_demo)
bc_policy = as_policy(bc_model)
print(f"BC final training loss: {final_loss:.2e}")
for name, pol in [("expert", expert_action), ("BC (30 demos)", bc_policy)]:
    sr, md = success_rate(pol)
    print(f"{name:16s} success = {sr:5.1%}   median final error = {md:.3f} m")

## 3-3. 분포 이동 (Covariate Shift) — "누가 팔을 툭 치면?"

50번째 스텝(남은 시간 30스텝)에서 관절을 무작위로 최대 ±1.5 rad 밀어 버립니다.
전문가는 언제든 현재 상태에서 다시 계산하니 회복합니다. BC 정책은 **시연에서 한 번도 본 적 없는 상태**에 놓이게 됩니다.

작은 실수 → 처음 보는 상태 → 더 큰 실수 → … 이렇게 오차가 눈덩이처럼 커지는 현상을 **compounding error** 라고 합니다.

In [ ]:
for name, pol in [("expert", expert_action), ("BC (30 demos)", bc_policy)]:
    sr, md = success_rate(pol, disturb=True)
    print(f"[disturbed] {name:16s} success = {sr:5.1%}   median final error = {md:.3f} m")

rng_vis = np.random.default_rng(7)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
for ax in axes:
    tgt = sample_target(rng_vis); seed = int(rng_vis.integers(1e6))
    for name, pol, c in [("expert", expert_action, "tab:green"), ("BC", bc_policy, "tab:red")]:
        _, qs, d = rollout(pol, tgt, np.random.default_rng(seed), disturb=True)
        hp = np.array([fk(q) for q in qs])
        ax.plot(hp[:, 0], hp[:, 1], color=c, label=f"{name} (err {d:.2f} m)")
        ax.plot(*hp[PUSH_T + 1], "x", color=c, ms=10)
    ax.plot(*tgt, "k*", ms=14); ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle(f"After a push at step {PUSH_T} (x = right after the push)")
plt.show()

### 🎬 밀친 뒤의 모습
BC가 실패한 장면 하나를 골라, 같은 장면에서 전문가와 BC를 나란히 재생합니다. 보라색 **PUSH!** 가 뜨는 순간 팔을 밀칩니다.
전문가는 다시 목표로 돌아가지만, BC는 처음 보는 자세에서 엉뚱하게 움직입니다.

In [ ]:
rng_pick = np.random.default_rng(11)
for _ in range(500):
    push_goal, push_seed = sample_target(rng_pick), int(rng_pick.integers(1e6))
    d_expert = rollout(expert_action, push_goal, np.random.default_rng(push_seed), disturb=True)[2]
    d_bc = rollout(bc_policy, push_goal, np.random.default_rng(push_seed), disturb=True)[2]
    if d_expert < SUCCESS_TOL and d_bc > 0.3:
        break


def pushed_path(policy):
    return rollout(policy, push_goal, np.random.default_rng(push_seed), disturb=True)[1]


show_arms([pushed_path(expert_action), pushed_path(bc_policy)], ["expert", "BC (30 demos)"],
          [push_goal] * 2, push_t=PUSH_T + 1)

## 3-4. DAgger — 실수한 상황을 데이터로 만들기

**DAgger (Dataset Aggregation)** 의 핵심:
1. 현재 정책으로 직접 로봇을 움직여 본다 (실수도 하고, 밀리기도 한다).
2. 그때 방문한 상태마다 **"전문가라면 여기서 어떻게 했을까?"** 를 물어 라벨을 붙인다.
3. 기존 데이터에 합쳐 다시 학습한다. → 반복

실제 현장에서는 로봇이 실수하는 순간 사람이 개입해 바로잡는 데이터(intervention data)를 모으는 방식으로 쓰입니다.

공정한 비교를 위해 **같은 횟수의 에피소드**만큼 시연을 더 모은 BC와도 비교합니다.

In [ ]:
def dagger(X0, Y0, iters=4, eps_per_iter=25, seed=1):
    rng = np.random.default_rng(seed)
    X, Y = X0.copy(), Y0.copy()
    model, _ = train_bc(X, Y)
    history = [success_rate(as_policy(model), disturb=True)[0]]
    for it in range(iters):
        pol = as_policy(model)
        newX, newY = [], []
        for _ in range(eps_per_iter):
            target = sample_target(rng)
            obs, qs, _ = rollout(pol, target, rng, disturb=True)   # 학습자 정책으로 실행
            newX.append(obs)
            newY.append(np.array([expert_action(q, target) for q in qs[:-1]]))  # 전문가가 라벨링
        X = np.concatenate([X] + newX); Y = np.concatenate([Y] + newY).astype(np.float32)
        model, _ = train_bc(X, Y, seed=it)
        history.append(success_rate(as_policy(model), disturb=True)[0])
        print(f"DAgger iter {it + 1}: dataset={len(X):5d}  disturbed success={history[-1]:5.1%}")
    return model, history


dagger_model, dagger_hist = dagger(X_demo, Y_demo)

X_more, Y_more = collect_demos(30 + 4 * 25, np.random.default_rng(0))
bc_more, _ = train_bc(X_more, Y_more)
sr_bc_more = success_rate(as_policy(bc_more), disturb=True)[0]
sr_expert = success_rate(expert_action, disturb=True)[0]
print(f"[disturbed] BC with 130 demos = {sr_bc_more:.0%}  |  DAgger (30 demos + 4 rounds) = {dagger_hist[-1]:.0%}")

plt.figure(figsize=(7, 4))
plt.plot(range(len(dagger_hist)), dagger_hist, "o-", label="DAgger")
plt.axhline(sr_bc_more, color="tab:red", ls="--", label=f"BC with 130 demos ({sr_bc_more:.0%})")
plt.axhline(sr_expert, color="tab:green", ls=":", label=f"expert ({sr_expert:.0%})")
plt.xlabel("DAgger iteration (0 = plain BC, 30 demos)"); plt.ylabel("success rate under push")
plt.ylim(0, 1.05); plt.grid(alpha=0.3); plt.legend()
plt.title("Learning to recover from mistakes")
plt.show()

### 🎬 같은 장면, DAgger 정책으로
3-3에서 BC가 실패한 바로 그 장면입니다. 실수한 상황을 데이터로 배운 DAgger 정책은 밀린 뒤에도 목표로 돌아오는지 보세요.

In [ ]:
show_arms([pushed_path(expert_action), pushed_path(bc_policy), pushed_path(as_policy(dagger_model))],
          ["expert", "BC (30 demos)", "DAgger"], [push_goal] * 3, push_t=PUSH_T + 1)

## 3-5. 미니 VLA — "빨간 공으로 가"

**VLA (Vision-Language-Action)** 모델은 카메라 영상과 언어 지시를 함께 받아 로봇 행동을 출력합니다.
여기서는 이미지 대신 "물체 목록(위치 + 색)"을 쓰는 **아주 작은 언어 조건 정책**을 만들어 핵심 원리를 봅니다.

- 책상 위에 빨강·초록·파랑 물체 3개가 무작위 위치·순서로 놓입니다.
- 정책은 **문장**을 읽고 어떤 물체로 갈지 스스로 골라야 합니다.
- 구조: 문장 → 단어 임베딩 평균(query) → 각 물체의 색(key)과 내적 → softmax **어텐션** → 고른 물체 위치 + 로봇 상태 → MLP → 행동
- 트랜스포머 기반 VLA의 cross-attention을 손바닥만 하게 줄인 모형입니다.

In [ ]:
COLOR_NAMES = ["빨간", "초록", "파란"]
TRAIN_TEMPLATES = ["{} 공으로 가", "{} 공을 잡아", "{} 물체로 팔을 뻗어", "{} 목표로 움직여"]
UNSEEN_TEMPLATE = "{} 쪽으로 천천히 이동해 줘"     # 학습 때 못 본 표현


def sample_scene(rng):
    while True:
        pts = [sample_target(rng) for _ in range(3)]
        if min(np.linalg.norm(pts[i] - pts[j]) for i in range(3) for j in range(i + 1, 3)) > 0.5:
            break
    colors = rng.permutation(3)                 # 물체 순서를 섞어서 "자리"로는 못 맞히게 함
    return pts, colors


def scene_features(pts, colors, q):
    """카메라가 본 장면을 흉내 낸 입력: 물체마다 (손끝 기준 상대 위치 dx, dy, 색 one-hot)"""
    hand = fk(q)
    feats = []
    for p, c in zip(pts, colors):
        feats += [p[0] - hand[0], p[1] - hand[1]] + list(np.eye(3)[c])
    return np.array(feats, dtype=np.float32)    # 3 objects x 5 = 15


class Vocab:
    def __init__(self, sentences):
        words = sorted({w for s in sentences for w in s.split()})
        self.idx = {w: i + 1 for i, w in enumerate(words)}  # 0 = <unk>

    def encode(self, s):
        ids = [self.idx[w] for w in s.split() if w in self.idx]   # 모르는 단어는 버림
        return ids or [0]                                          # 아는 단어가 하나도 없으면 <unk>


train_sentences = [t.format(c) for t in TRAIN_TEMPLATES for c in COLOR_NAMES]
vocab = Vocab(train_sentences)
print("vocab size:", len(vocab.idx) + 1, "|", list(vocab.idx)[:8], "...")


class MiniVLA(nn.Module):
    """언어 임베딩(query)으로 물체(key)에 어텐션 → 고른 물체 위치 + 로봇 상태 → 행동.
    실제 VLA의 트랜스포머 cross-attention을 아주 작게 줄인 구조입니다."""

    def __init__(self, vocab_size, emb=16, hidden=128):
        super().__init__()
        self.lang = nn.EmbeddingBag(vocab_size, emb, mode="mean")   # 언어 인코더 (아주 작은!)
        self.obj_key = nn.Linear(3, emb)                              # 물체의 색 → key
        self.policy = nn.Sequential(nn.Linear(6 + 2, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, 2), nn.Tanh())

    def attend(self, scene, tokens, offsets):
        objs = scene.view(-1, 3, 5)                                   # (B, 물체 3개, [dx, dy, 색 3])
        query = self.lang(tokens, offsets)                            # (B, emb)
        keys = self.obj_key(objs[..., 2:])                            # (B, 3, emb)
        att = torch.softmax((keys @ query.unsqueeze(-1)).squeeze(-1), dim=-1)  # (B, 3)
        return att, objs[..., :2]

    def forward(self, robot, scene, tokens, offsets):
        att, rel_pos = self.attend(scene, tokens, offsets)
        goal_rel = (att.unsqueeze(-1) * rel_pos).sum(1)               # 어텐션으로 고른 물체의 상대 위치
        return self.policy(torch.cat([robot, goal_rel], dim=-1)) * MAX_DQ


def robot_state(q):
    return np.concatenate([np.cos(q), np.sin(q), fk(q)]).astype(np.float32)


def vla_rollout(act_fn, pts, colors, goal_color, rng, disturb=False):
    q = HOME + rng.normal(0, 0.05, 2)
    goal = pts[list(colors).index(goal_color)]
    states = []
    for t in range(T):
        if disturb and t == PUSH_T:
            q = q + rng.uniform(-PUSH, PUSH, 2)
        states.append((robot_state(q), scene_features(pts, colors, q), q.copy(), goal))
        q = q + np.clip(act_fn(q), -MAX_DQ, MAX_DQ)
    dists = [np.linalg.norm(fk(q) - p) for p in pts]
    return states, np.linalg.norm(fk(q) - goal), int(np.argmin(dists)) == list(colors).index(goal_color)


def collect_vla_data(n_eps, rng, policy=None):
    R, S, toks, A = [], [], [], []
    for _ in range(n_eps):
        pts, colors = sample_scene(rng)
        gc = int(rng.integers(3))
        sent = rng.choice(TRAIN_TEMPLATES).format(COLOR_NAMES[gc])
        goal = pts[list(colors).index(gc)]
        act = (lambda q: expert_action(q, goal)) if policy is None else (lambda q: policy(q, pts, colors, sent))
        states, _, _ = vla_rollout(act, pts, colors, gc, rng, disturb=policy is not None)
        for rs, sf, q, g in states:
            R.append(rs); S.append(sf); toks.append(vocab.encode(sent)); A.append(expert_action(q, g))
    return R, S, toks, A


def to_batch(token_lists):
    offsets = torch.tensor([0] + [len(t) for t in token_lists[:-1]]).cumsum(0)
    return torch.tensor([w for t in token_lists for w in t]), offsets


def train_vla(data, steps=2500, seed=0):
    torch.manual_seed(seed)
    R, S, toks, A = data
    R, S, A = torch.tensor(np.array(R)), torch.tensor(np.array(S)), torch.tensor(np.array(A, dtype=np.float32))
    model = MiniVLA(len(vocab.idx) + 1)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(steps):
        idx = torch.randint(0, len(R), (256,))
        tk, off = to_batch([toks[i] for i in idx.tolist()])
        loss = ((model(R[idx], S[idx], tk, off) - A[idx]) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return model


def vla_policy(model):
    def act(q, pts, colors, sentence):
        tk, off = to_batch([vocab.encode(sentence)])
        sf = torch.from_numpy(scene_features(pts, colors, q))[None]
        with torch.no_grad():
            return model(torch.from_numpy(robot_state(q))[None], sf, tk, off)[0].numpy()
    return act


# 시연 400회 수집 → 행동 복제로 학습
rng = np.random.default_rng(0)
data = collect_vla_data(400, rng)
vla = train_vla(data)
print("training samples:", len(data[0]))

# 정책이 "어느 물체를 보고 있는지" 어텐션 가중치로 확인
pts, colors = sample_scene(np.random.default_rng(5))
names = ["red", "green", "blue"]
print("scene objects (slot order):", [names[c] for c in colors])
for sent in ["빨간 공으로 가", "초록 공을 잡아", "파란 물체로 팔을 뻗어"]:
    tk, off = to_batch([vocab.encode(sent)])
    att, _ = vla.attend(torch.from_numpy(scene_features(pts, colors, HOME))[None], tk, off)
    print(f"  '{sent}' -> attention {np.round(att[0].detach().numpy(), 2)}")

In [ ]:
def eval_vla(model, sentence_fn, n=100, seed=999):
    rng = np.random.default_rng(seed)
    act = vla_policy(model)
    ok_reach, ok_choice = 0, 0
    for _ in range(n):
        pts, colors = sample_scene(rng)
        gc = int(rng.integers(3))
        sent = sentence_fn(gc)
        _, d, right = vla_rollout(lambda q: act(q, pts, colors, sent), pts, colors, gc, rng)
        ok_reach += d < SUCCESS_TOL; ok_choice += right
    return ok_reach / n, ok_choice / n


tests = {
    "학습한 표현   '파란 공으로 가'": lambda c: TRAIN_TEMPLATES[0].format(COLOR_NAMES[c]),
    "처음 보는 표현 '파란 쪽으로 천천히 이동해 줘'": lambda c: UNSEEN_TEMPLATE.format(COLOR_NAMES[c]),
    "처음 보는 색 단어 '파랑 공으로 가'": lambda c: TRAIN_TEMPLATES[0].format(["빨강", "초록색", "파랑"][c]),
}
for name, fn in tests.items():
    reach, choice = eval_vla(vla, fn)
    print(f"{name:40s} 올바른 물체 선택 {choice:5.1%} | 도달 성공 {reach:5.1%}")

### 🎬 말로 지시하기 영상
같은 책상에 세 가지 지시를 줍니다. 점선 원이 지시한 물체이고, 그림 제목의 **attention** 은 정책이 빨강(R)·초록(G)·파랑(B) 물체에 각각 둔 관심(어텐션 가중치)입니다.

In [ ]:
COLOR_PLOT = ["tab:red", "tab:green", "tab:blue"]
vla_pts, vla_colors = sample_scene(np.random.default_rng(5))
commands = [("빨간 공으로 가", 0, "학습한 표현"),
            ("초록 쪽으로 천천히 이동해 줘", 1, "처음 보는 표현"),
            ("파랑 공으로 가", 2, "처음 보는 색 단어")]
display(HTML("<br>".join(f"<b>{i + 1}.</b> “{s}” ({note})" for i, (s, _, note) in enumerate(commands))))

act = vla_policy(vla)
paths, titles, goals = [], [], []
for i, (sentence, goal_color, _) in enumerate(commands):
    q = HOME.copy(); qs = [q.copy()]
    for _ in range(T):
        q = q + np.clip(act(q, vla_pts, vla_colors, sentence), -MAX_DQ, MAX_DQ)
        qs.append(q.copy())
    tk, off = to_batch([vocab.encode(sentence)])
    att = vla.attend(torch.from_numpy(scene_features(vla_pts, vla_colors, HOME))[None], tk, off)[0][0].detach().numpy()
    by_color = {["red", "green", "blue"][c]: a for c, a in zip(vla_colors, att)}
    paths.append(np.array(qs)); goals.append(vla_pts[list(vla_colors).index(goal_color)])
    titles.append(f"command {i + 1}\nattention  " + " / ".join(f"{k[0].upper()} {by_color[k]:.0%}" for k in ["red", "green", "blue"]))

show_arms(paths, titles, goals, objects=[[(p, COLOR_PLOT[c]) for p, c in zip(vla_pts, vla_colors)]] * 3)

**결과 해석**
- 학습한 문장: 색 단어를 보고 올바른 물체를 골라 도달합니다. → 언어가 행동을 **조건화(conditioning)** 합니다.
- 처음 보는 표현: 색 단어("파란")만 알면 나머지 낯선 단어는 버려도 대체로 동작합니다.
- 처음 보는 색 단어("파랑", "빨강"): 사전에 없는 단어라 색을 구분하지 못합니다 → 정답률이 크게 떨어집니다 (3개 중 하나를 고르는 문제라 우연히 맞히는 수준은 약 33%).

실제 VLA(RT-2, OpenVLA, π0, GR00T 등)가 **인터넷 규모로 사전학습된 VLM** 위에 만들어지는 이유가 바로 이것입니다.
"파랑 = 파란 = blue = 하늘색 비슷한 것"이라는 상식은 로봇 데이터만으로는 배우기 어렵고, 웹 데이터에서 빌려 와야 합니다.

### ✍️ 직접 해 보기 (도전 과제)
1. 3-2에서 시연 수를 5, 10, 100개로 바꿔 BC 성공률 곡선을 그려 보세요. "데이터 스케일링"을 체감할 수 있습니다.
2. 3-5의 평가에 외란(`disturb=True`)을 넣어 보고, 3-4의 DAgger를 미니 VLA에도 적용해 보세요.
3. `Vocab` 대신 사전학습된 문장 임베딩(예: `sentence-transformers` 다국어 모델)을 쓰면 "파랑"도 맞힐 수 있을까요?
4. 행동을 한 스텝이 아니라 **여러 스텝 묶음(action chunk)** 으로 예측하도록 바꿔 보세요 (ACT의 아이디어).

## 정리
- **행동 복제**: 시연을 지도학습으로 따라 하기. 단순하고 강력하지만 **분포 이동**에 취약.
- **DAgger / 개입 데이터**: 정책이 실제로 가는 상태에서 전문가 라벨을 모아 약점을 메움.
- **VLA**: 언어(와 영상)로 조건화된 정책. 일반화의 열쇠는 **사전학습된 지식 + 다양한 로봇 데이터**.